# YExtract: single-transition weighted projection workflow

Read one transition CSV, build reusable 2D densities, draw pcolormesh figures, compute weighted 1D projections, and export arrays plus summaries.

## Imports

In [ ]:
from pathlib import Path

import matplotlib as plt
import pandas as pd
import matplotlib.pyplot as plt

from YAnalysis import TransitionData, projection_summary_to_row
import YPlot

## User configuration

In [ ]:
CSV_PATH = Path(r"path\to\file_name.csv.gz")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

SIGMA_GATE = (0.2, 0.4)
GAUSSIAN_MU = 0.30
GAUSSIAN_SIGMA = 0.02

## Load transition CSV

In [ ]:
print(f"CSV_PATH: {CSV_PATH}")
data = TransitionData(CSV_PATH)
data.df.head()

## Build 2D densities

In [ ]:
delta_sigma = data.make_delta_sigma_density(normalize=True)
theta_sigma = data.make_theta_sigma_density(normalize=True)
pado = data.make_pado_density(normalize=False)

delta_sigma.z.shape, theta_sigma.z.shape, pado.z.shape

## Plot P-ADO, delta-sigma/I, theta-sigma/I

In [ ]:
fig, ax, mesh = YPlot.plot_pado_scatter(pado, cmap=YPlot.get_root_kbird_cmap())
YPlot.apply_style(ax, preset="journal")
ax.set_xlabel(r"$R_\mathrm{ADO}$")
ax.set_ylabel(r"$P$")
# YPlot.save_figure(fig, OUTPUT_DIR / "P-ADO.png")
plt.show()

In [ ]:
fig, ax, mesh = YPlot.plot_density2d(delta_sigma, cmap="viridis")
YPlot.apply_style(ax, preset="journal")
# YPlot.save_figure(fig, OUTPUT_DIR / "delta_sigma.png")
plt.show()

In [ ]:
fig, ax, mesh = YPlot.plot_density2d(theta_sigma, cmap="viridis")
YPlot.apply_style(ax, preset="journal")
# YPlot.save_figure(fig, OUTPUT_DIR / "theta_sigma.png")
plt.show()

## Constant full-range projection

In [ ]:
proj_delta_all = delta_sigma.project(
    target_axis="x",
    weight_kind="constant",
    weight_range=None,
    normalize=True,
    evaluate=True,
)

print(proj_delta_all.format_summary())

fig, ax, line = YPlot.plot_projection1d(
    proj_delta_all,
    add_eq_lines=True,
    add_hdi_spans=True,
)
YPlot.apply_style(ax, preset="journal")
# YPlot.save_figure(fig, OUTPUT_DIR / "projection_delta_gate.png")
plt.show()

In [ ]:
proj_theta_all = theta_sigma.project(
    target_axis="x",
    weight_kind="constant",
    weight_range=None,
    normalize=True,
    evaluate=True,
)

print(proj_theta_all.format_summary())

fig, ax, line = YPlot.plot_projection1d(
    proj_theta_all,
    add_eq_lines=True,
    add_hdi_spans=True,
)
YPlot.apply_style(ax, preset="journal")
# YPlot.save_figure(fig, OUTPUT_DIR / "projection_delta_gate.png")
plt.show()

## Constant sigma-gated projection

In [ ]:
proj_delta_gate = delta_sigma.project(
    target_axis="x",
    weight_kind="constant",
    weight_range=SIGMA_GATE,
    normalize=True,
    evaluate=True,
)

print(proj_delta_gate.format_summary())

fig, ax, line = YPlot.plot_projection1d(
    proj_delta_gate,
    add_eq_lines=True,
    add_hdi_spans=True,
)
YPlot.apply_style(ax, preset="journal")
# YPlot.save_figure(fig, OUTPUT_DIR / "projection_delta_gate.png")
plt.show()

In [ ]:
proj_theta_gate = theta_sigma.project(
    target_axis="x",
    weight_kind="constant",
    weight_range=SIGMA_GATE,
    normalize=True,
    evaluate=True,
)

print(proj_theta_gate.format_summary())

fig, ax, line = YPlot.plot_projection1d(
    proj_theta_gate,
    add_eq_lines=True,
    add_hdi_spans=True,
)
YPlot.apply_style(ax, preset="journal")
# YPlot.save_figure(fig, OUTPUT_DIR / "projection_theta_gate.png")
plt.show()

## Gaussian weighted projection

In [ ]:
proj_delta_gauss = delta_sigma.project(
    target_axis="x",
    weight_kind="gaussian",
    gaus_mu=GAUSSIAN_MU,
    gaus_sigma=GAUSSIAN_SIGMA,
    weight_range=None,
    normalize=True,
    evaluate=True,
)

print(proj_delta_gauss.format_summary())

fig, ax, line = YPlot.plot_projection1d(
    proj_delta_gauss,
    add_eq_lines=True,
    add_hdi_spans=True,
)
YPlot.apply_style(ax, preset="journal")
# YPlot.save_figure(fig, OUTPUT_DIR / "projection_delta_gaussian.png")
plt.show()

In [ ]:
proj_theta_gauss = theta_sigma.project(
    target_axis="x",
    weight_kind="gaussian",
    gaus_mu=GAUSSIAN_MU,
    gaus_sigma=GAUSSIAN_SIGMA,
    weight_range=None,
    normalize=True,
    evaluate=True,
)

print(proj_theta_gauss.format_summary())

fig, ax, line = YPlot.plot_projection1d(
    proj_theta_gauss,
    add_eq_lines=True,
    add_hdi_spans=True,
)
YPlot.apply_style(ax, preset="journal")
# YPlot.save_figure(fig, OUTPUT_DIR / "projection_theta_gaussian.png")
plt.show()

## Export projection arrays and summary

In [ ]:
proj_delta_gate.to_dataframe().to_csv(OUTPUT_DIR / "projection_delta_gate.csv", index=False)
proj_delta_gate.weight_to_dataframe().to_csv(OUTPUT_DIR / "projection_delta_gate_weight.csv", index=False)

summary_rows = []
projection_items = [
    ("delta_all", proj_delta_all, "constant full-range"),
    ("delta_gate", proj_delta_gate, f"constant gate {SIGMA_GATE}"),
    ("delta_gaussian", proj_delta_gauss, f"gaussian gaus_mu={GAUSSIAN_MU}, gaus_sigma={GAUSSIAN_SIGMA}"),
]

for label, proj, description in projection_items:
    row = projection_summary_to_row(
        proj,
        transition_name=data.name,
        quantity=proj.coord_col,
        gate_label=label,
        label=label,
    )
    row["description"] = description
    row["summary_text"] = proj.format_summary()
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

summary_wide_df = summary_df.drop(
    columns=["hdi_text", "summary_text"],
    errors="ignore",
)

summary_wide_df.to_csv(
    OUTPUT_DIR / "projection_summaries_wide.csv",
    index=False,
)

summary_text_df = summary_df[["label", "description", "summary_text"]]
summary_text_df.to_csv(OUTPUT_DIR / "projection_summaries_text.csv", index=False)

display_cols = [
    "label",
    "description",
    "quantity",
    "weight_kind",
    "gate_type",
    "gate_range",
    "gaus_mu",
    "gaus_sigma",
    "mean",
    "std",
    "eq_q50",
    "eq_err_minus",
    "eq_err_plus",
    "hdi_interval_count",
    "hdi_interval",
    "hdi_interval_mass",
    "hdi_interval_mass_fraction",
    "hdi_interval_mass_rank",
]
summary_df[display_cols]